# Classificando o motivo das avaliações negativas com IA generativa (Gemini)

Na análise exploratória (`01_analise_exploratoria.ipynb`), **57% das avaliações negativas vieram de pedidos
entregues no prazo**: o atraso não explica a maior parte da insatisfação. Para descobrir o que explica, este notebook
usa o **Gemini** para ler os comentários dos clientes e classificar o motivo principal de cada reclamação.

**Como funciona**
1. Seleciona uma amostra de avaliações negativas (nota 1 ou 2) com comentário escrito.
2. Envia os comentários em lotes para o Gemini, com uma lista fechada de motivos e resposta em JSON estruturado.
3. Compara os motivos entre pedidos **entregues no prazo**, **com atraso** e **não entregues**.

**Para rodar no Google Colab**
- Crie uma chave em [Google AI Studio](https://aistudio.google.com/apikey) e salve nos *Secrets* do Colab com o nome
  `key_gemini` (ícone de chave, na barra lateral). **Nunca cole a chave no código.**
- Tenha em mãos `olist_order_reviews_dataset.csv` e `olist_orders_dataset.csv`
  ([dataset no Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)).

In [ ]:
!pip install -q google-genai

## 1. Configuração

In [ ]:
import os
import time
import json
from enum import Enum

import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

MODELO = "gemini-3-flash-preview"   # troque se o modelo não estiver disponível na sua conta
AMOSTRA_POR_GRUPO = {"No prazo": 300, "Com atraso": 150, "Não entregue": 150}
TAMANHO_LOTE = 30                   # comentários por chamada à API
PAUSA_SEGUNDOS = 6                  # respeita o limite de requisições do plano gratuito
SEMENTE = 42                        # amostra reproduzível

EM_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
PASTA = "." if EM_COLAB else ".."
DADOS, IMG = f"{PASTA}/dados", f"{PASTA}/img"
os.makedirs(DADOS, exist_ok=True)
os.makedirs(IMG, exist_ok=True)
ARQ_RESULTADO = f"{DADOS}/classificacao_gemini.csv"

In [ ]:
from google import genai
from google.genai import types

if EM_COLAB:
    from google.colab import userdata
    CHAVE = userdata.get("key_gemini")
else:
    CHAVE = os.environ["GEMINI_API_KEY"]

client = genai.Client(api_key=CHAVE)

## 2. Dados

In [ ]:
if EM_COLAB:
    from google.colab import files
    print("Envie olist_order_reviews_dataset.csv e olist_orders_dataset.csv")
    for nome, conteudo in files.upload().items():
        with open(f"{DADOS}/{nome}", "wb") as f:
            f.write(conteudo)

reviews = pd.read_csv(f"{DADOS}/olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
orders = pd.read_csv(f"{DADOS}/olist_orders_dataset.csv",
                     parse_dates=["order_delivered_customer_date", "order_estimated_delivery_date"])

reviews = (reviews.sort_values("review_answer_timestamp")
           .drop_duplicates("order_id", keep="last")
           .drop_duplicates("review_id", keep="last"))
base = reviews.merge(orders, on="order_id", how="left")

entregue = base["order_status"].eq("delivered") & base["order_delivered_customer_date"].notna()
atraso = (base["order_delivered_customer_date"].dt.normalize()
          - base["order_estimated_delivery_date"].dt.normalize()).dt.days
base["grupo"] = "Não entregue"
base.loc[entregue & (atraso <= 0), "grupo"] = "No prazo"
base.loc[entregue & (atraso > 0), "grupo"] = "Com atraso"

base["comentario"] = ((base["review_comment_title"].fillna("") + ". " + base["review_comment_message"].fillna(""))
                      .str.replace(r"\s+", " ", regex=True).str.strip(" ."))
negativas = base[(base["review_score"] <= 2) & base["review_comment_message"].fillna("").str.strip().ne("")]
print(f"Avaliações negativas com comentário: {len(negativas):,}")
negativas["grupo"].value_counts()

### Amostra

Uma amostra estratificada por grupo mantém o custo baixo (cabe no plano gratuito) e garante comentários suficientes
de cada situação para comparar os motivos. Com 300 comentários, a margem de erro de uma proporção fica em torno
de ±6 pontos percentuais (95% de confiança).

In [ ]:
amostra = pd.concat([
    negativas[negativas["grupo"] == g].sample(n, random_state=SEMENTE)
    for g, n in AMOSTRA_POR_GRUPO.items()
]).reset_index(drop=True)
amostra["id"] = amostra.index
amostra["comentario"] = amostra["comentario"].str.slice(0, 400)
amostra["grupo"].value_counts()

## 3. Classificação com o Gemini

A IA escolhe **um motivo principal** dentro de uma lista fechada, o que torna o resultado comparável e fácil de
agregar. A resposta vem em JSON validado por um esquema (Pydantic), sem precisar interpretar texto livre.

In [ ]:
class Motivo(str, Enum):
    ATRASO = "Atraso na entrega"
    NAO_RECEBIDO = "Produto não recebido"
    INCOMPLETO = "Pedido incompleto"
    ERRADO = "Produto errado ou diferente do anúncio"
    DEFEITO = "Produto com defeito ou danificado"
    QUALIDADE = "Qualidade abaixo do esperado"
    ATENDIMENTO = "Atendimento e pós-venda"
    OUTRO = "Outro ou sem motivo claro"


class Classificacao(BaseModel):
    id: int
    motivo: Motivo
    resumo: str = Field(description="O problema em até 10 palavras")


INSTRUCOES = '''Você analisa avaliações negativas de clientes de um e-commerce brasileiro.
Para cada comentário, escolha UM motivo principal da insatisfação:
- Atraso na entrega: o produto chegou, mas depois do esperado, ou o cliente reclama da demora.
- Produto não recebido: o cliente diz que o produto não chegou.
- Pedido incompleto: chegaram só parte dos itens ou faltou alguma peça.
- Produto errado ou diferente do anúncio: veio outro produto, cor, tamanho, modelo ou marca.
- Produto com defeito ou danificado: quebrado, não funciona, avariado no transporte.
- Qualidade abaixo do esperado: chegou certo e funcionando, mas o material ou acabamento decepcionou.
- Atendimento e pós-venda: dificuldade de contato, troca, devolução, reembolso ou cancelamento.
- Outro ou sem motivo claro: o texto não permite identificar o motivo.
Responda apenas com a lista JSON, mantendo o mesmo id de cada comentário.'''


def classificar_lote(lote, tentativas=5):
    textos = "\n".join(f"[{r.id}] {r.comentario}" for r in lote.itertuples())
    for tentativa in range(tentativas):
        try:
            resposta = client.models.generate_content(
                model=MODELO,
                contents=f"{INSTRUCOES}\n\nComentários:\n{textos}",
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=list[Classificacao],
                    temperature=0,
                ),
            )
            return [c.model_dump(mode="json") for c in resposta.parsed]
        except Exception as erro:
            espera = 20 * (tentativa + 1)
            print(f"  erro ({type(erro).__name__}: {str(erro)[:80]}). Nova tentativa em {espera}s")
            time.sleep(espera)
    raise RuntimeError("Não foi possível classificar o lote após várias tentativas.")

O loop salva o resultado a cada lote. Se a cota da API acabar no meio, é só rodar a célula de novo no dia seguinte:
ela continua de onde parou.

In [ ]:
feitos = pd.read_csv(ARQ_RESULTADO) if os.path.exists(ARQ_RESULTADO) else pd.DataFrame(columns=["id", "motivo", "resumo"])
pendentes = amostra[~amostra["id"].isin(feitos["id"])]
print(f"Já classificados: {len(feitos)} | Pendentes: {len(pendentes)}")

for inicio in range(0, len(pendentes), TAMANHO_LOTE):
    lote = pendentes.iloc[inicio:inicio + TAMANHO_LOTE]
    novos = pd.DataFrame(classificar_lote(lote))
    novos = novos[novos["id"].isin(lote["id"])].drop_duplicates("id")
    feitos = pd.concat([feitos, novos], ignore_index=True)
    feitos.to_csv(ARQ_RESULTADO, index=False)
    print(f"Lote {inicio // TAMANHO_LOTE + 1}: {len(novos)}/{len(lote)} classificados (total {len(feitos)})")
    time.sleep(PAUSA_SEGUNDOS)

resultado = amostra.merge(feitos, on="id", how="inner")
print(f"Classificados: {len(resultado)} de {len(amostra)}")

## 4. Resultados

In [ ]:
tabela = (pd.crosstab(resultado["motivo"], resultado["grupo"], normalize="columns") * 100).round(1)
tabela = tabela[["No prazo", "Com atraso", "Não entregue"]].sort_values("No prazo", ascending=False)
tabela

In [ ]:
no_prazo = (resultado[resultado["grupo"] == "No prazo"]["motivo"]
            .value_counts(normalize=True).sort_values() * 100)

fig, ax = plt.subplots(figsize=(10, 5.8))
fig.subplots_adjust(left=0.36, right=0.95, top=0.80, bottom=0.10)
ys = range(len(no_prazo))
ax.barh(list(ys), no_prazo.values, height=0.62, color="#2a78d6", zorder=3)
for y, v in zip(ys, no_prazo.values):
    ax.text(v + 0.6, y, f"{v:.0f}%", va="center", fontsize=11, color="#0b0b0b")
ax.set_yticks(list(ys), no_prazo.index, fontsize=11)
ax.set_xlim(0, max(no_prazo.values) * 1.15)
ax.xaxis.grid(True, color="#e6e5e1", zorder=0)
ax.xaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
for lado in ["top", "right"]:
    ax.spines[lado].set_visible(False)
ax.tick_params(length=0)
fig.text(0.04, 0.95, "O que causa a insatisfação quando a entrega chega no prazo",
         fontsize=16, fontweight="bold", color="#0b0b0b", va="top")
fig.text(0.04, 0.885, f"Motivo principal de {int(len(resultado[resultado.grupo == 'No prazo']))} avaliações negativas "
         "de pedidos entregues no prazo, classificado pelo Gemini", fontsize=11.5, color="#52514e", va="top")
fig.text(0.04, 0.02, "Fonte: Olist Brazilian E-Commerce Public Dataset. Classificação: Gemini. Análise: Matheus Cadoni.",
         fontsize=9.5, color="#8a8984")
fig.savefig(f"{IMG}/5_motivos_no_prazo.png", dpi=160)
plt.show()

## 5. Conferência manual

Toda classificação feita por IA precisa ser checada. Leia os exemplos abaixo e anote quantos você concorda.
Uma taxa de acerto acima de 85% a 90% indica que a classificação é confiável para a análise.

In [ ]:
pd.set_option("display.max_colwidth", 120)
resultado.sample(25, random_state=SEMENTE)[["grupo", "comentario", "motivo", "resumo"]]

## 6. Conclusões

> Preencha depois de rodar, com base na tabela e no gráfico acima. Perguntas para guiar:
> - Entre os pedidos **entregues no prazo**, qual é o motivo mais comum? Ele é de logística ou de produto?
> - Quantos clientes com pedido "entregue" dizem que **não receberam** ou que o **pedido veio incompleto**? Isso
>   pode indicar falha no registro de entrega ou em pedidos com vários itens.
> - O que a empresa deveria atacar primeiro para reduzir as avaliações negativas?